# Pré-processamento

## 1. Importações e Carregamento dos Dados

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

data_path = Path("../data/creditcard.csv")
df = pd.read_csv(data_path)
print(f"Dataset carregado: {df.shape[0]:,} transações, {df.shape[1]} colunas")

Dataset carregado: 284,807 transações, 31 colunas


## 2. Separação de Features (X) e Target (y)

In [2]:
X = df.drop(columns=["Class"])
y = df["Class"]

print(f"X: {X.shape}  |  y: {y.shape}")
print(f"\nDistribuição das classes:\n{y.value_counts().to_string()}")
print(f"\nProporção de fraudes: {y.mean():.4%}")

X: (284807, 30)  |  y: (284807,)

Distribuição das classes:
Class
0    284315
1       492

Proporção de fraudes: 0.1727%


## 3. Divisão em Treino e Teste (80/20 Estratificado)

A divisão é realizada **antes** da normalização para evitar *data leakage*: o `StandardScaler` será ajustado exclusivamente nos dados de treino e depois aplicado ao conjunto de teste.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Treino : {X_train.shape[0]:,} amostras ({X_train.shape[0] / len(X):.0%})")
print(f"Teste  : {X_test.shape[0]:,} amostras ({X_test.shape[0] / len(X):.0%})")
print(f"\nDistribuição no treino:\n{y_train.value_counts().to_string()}")
print(f"\nDistribuição no teste:\n{y_test.value_counts().to_string()}")

Treino : 227,845 amostras (80%)
Teste  : 56,962 amostras (20%)

Distribuição no treino:
Class
0    227451
1       394

Distribuição no teste:
Class
0    56864
1       98


## 4. Verificação e Normalização de Escala

As colunas `V1`–`V28` são o resultado de uma **PCA** aplicada no Kaggle antes da publicação do dataset: já estão centradas (média ≈ 0) e portanto **não precisam de normalização adicional**.

`Time` e `Amount` são variáveis brutas e precisam de escalonamento. O `StandardScaler` é ajustado **apenas no conjunto de treino** (`fit_transform`) e depois aplicado ao teste (`transform`), evitando vazamento de informação (*data leakage*).

In [4]:
print("Antes da normalização — Time e Amount (treino):")
print(X_train[["Time", "Amount"]].describe().round(2))

scaler = StandardScaler()
cols_to_scale = ["Time", "Amount"]

X_train = X_train.copy()
X_test = X_test.copy()
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

print("\nApós normalização — Time e Amount (treino):")
print(X_train[["Time", "Amount"]].describe().round(3))

Antes da normalização — Time e Amount (treino):
            Time     Amount
count  227845.00  227845.00
mean    94885.09      88.18
std     47488.42     250.72
min         0.00       0.00
25%     54228.00       5.64
50%     84805.00      22.00
75%    139364.00      77.49
max    172792.00   25691.16

Após normalização — Time e Amount (treino):
             Time      Amount
count  227845.000  227845.000
mean       -0.000      -0.000
std         1.000       1.000
min        -1.998      -0.352
25%        -0.856      -0.329
50%        -0.212      -0.264
75%         0.937      -0.043
max         1.641     102.117


## 5. Balanceamento das Classes com SMOTE

O SMOTE é aplicado **apenas no conjunto de treino** e será utilizado pelos modelos supervisionados (**Random Forest** e **XGBoost**). Os modelos não supervisionados não dependem de classes balanceadas.

In [5]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

y_train_smote_series = pd.Series(y_train_smote)
print(f"Antes do SMOTE : {y_train.value_counts().to_dict()}")
print(f"Após o SMOTE   : {y_train_smote_series.value_counts().to_dict()}")
print(f"\nX_train_smote shape: {X_train_smote.shape}")

Antes do SMOTE : {0: 227451, 1: 394}
Após o SMOTE   : {0: 227451, 1: 227451}

X_train_smote shape: (454902, 30)


## 6. Preparação para Modelos Não Supervisionados

| Modelo            | Dados de treino   | Estratégia                                                               |
|-------------------|-------------------|--------------------------------------------------------------------------|
| Random Forest     | `X_train_smote`   | Supervisionado; classes balanceadas via SMOTE                            |
| XGBoost           | `X_train_smote`   | Supervisionado; classes balanceadas via SMOTE                            |
| Isolation Forest  | `X_train`         | Não supervisionado — aprende a isolar anomalias pelo caminho de partição |
| Autoencoder (MLP) | `X_train_normal`  | Treinado só em transações normais; fraude = alta reconstrução            |

Todos os modelos são avaliados em `X_test` / `y_test`.

In [6]:
X_train_normal = X_train[y_train == 0]

print(f"X_train        : {X_train.shape}  — Isolation Forest")
print(f"X_train_normal : {X_train_normal.shape}  — Autoencoder (MLPRegressor)")
print(f"X_train_smote  : {X_train_smote.shape}  — Random Forest / XGBoost")
print(f"X_test         : {X_test.shape}  — avaliação de todos os modelos")

X_train        : (227845, 30)  — Isolation Forest
X_train_normal : (227451, 30)  — Autoencoder (MLPRegressor)
X_train_smote  : (454902, 30)  — Random Forest / XGBoost
X_test         : (56962, 30)  — avaliação de todos os modelos


**Considerações sobre o Pré-processamento**

Ao final do pré-processamento temos três versões do conjunto de treino prontas para os 4 modelos:
- `X_train_smote` / `y_train_smote` → Random Forest e XGBoost
- `X_train` (full) → Isolation Forest
- `X_train_normal` (Class=0 apenas) → Autoencoder (MLPRegressor)

# Construção do Meta-modelo